In [ ]:
import torch, tensorflow as tf
print('PyTorch CUDA:', torch.cuda.is_available())
print('TF GPU:', tf.config.list_physical_devices('GPU'))

PyTorch CUDA: True
TF GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Cell 2 — Install HuggingFace stack
!pip install -q datasets transformers huggingface_hub
!pip install -q transformers datasets accelerate seqeval
!pip install -q spacy && python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 124.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Cell 3 — HuggingFace login (only if using gated datasets)
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# ── Cell 1: Build urgency-labelled dataset ──────────────────────
from datasets import Dataset, DatasetDict
# Synthetic complaint-urgency pairs (add more for better accuracy)
# ── Cell 1: Build urgency-labelled dataset ──────────────────────
complaints = [
    # CRITICAL (label 0) — 20 samples
    ('Severe chest pain since 2 hours with breathlessness and sweating', 0),
    ('Patient unconscious, not responding to stimuli', 0),
    ('Sudden stroke, left side paralysis, face drooping', 0),
    ('Seizure lasting more than 5 minutes, still shaking', 0),
    ('Crushing chest pain radiating to left arm and jaw', 0),
    ('Patient not breathing, no pulse, needs CPR immediately', 0),
    ('Severe allergic reaction with throat swelling and wheezing', 0),
    ('Massive uncontrolled bleeding from deep abdominal wound', 0),
    ('High fever with stiff neck, sensitivity to light, rash', 0),
    ('Sudden worst headache of life with vision loss', 0),
    ('Acute breathlessness, lips turning blue, oxygen dropping', 0),
    ('Diabetic patient unconscious, blood sugar critically low', 0),
    ('Severe burns covering chest and both arms from fire', 0),
    ('Multiple trauma from road accident, bleeding from head', 0),
    ('Suspected poisoning, patient vomiting blood, confused', 0),
    ('Heart attack symptoms, ECG showing ST elevation', 0),
    ('Respiratory failure, unable to speak full sentences', 0),
    ('Sudden complete loss of vision in both eyes', 0),
    ('Severe electric shock, patient in and out of consciousness', 0),
    ('Gunshot wound to abdomen, heavy internal bleeding suspected', 0),

    # MODERATE (label 1) — 20 samples
    ('High fever 103F with vomiting and dehydration since morning', 1),
    ('Moderate abdominal pain, nausea, dizziness since morning', 1),
    ('Persistent headache with blurred vision and mild weakness', 1),
    ('Allergic reaction with mild swelling, no breathing difficulty', 1),
    ('Continuous vomiting for 6 hours, unable to keep water down', 1),
    ('Chest pain only on deep breathing, mild shortness of breath', 1),
    ('Fever 102F with severe body ache and joint pain', 1),
    ('Moderate burn on hand and forearm from kitchen accident', 1),
    ('Dizziness and fainting spell, now conscious but very weak', 1),
    ('Urinary infection with severe burning sensation and back pain', 1),
    ('Asthma attack, partially relieved by inhaler, still wheezing', 1),
    ('Suspected fracture in wrist after fall, severe pain swelling', 1),
    ('Severe migraine with nausea, photosensitivity, since 12 hours', 1),
    ('Appendicitis-like pain in lower right abdomen, worsening', 1),
    ('Animal bite on leg, wound deep, tetanus status unknown', 1),
    ('Moderate dehydration after diarrhea for 24 hours, weak', 1),
    ('Chest tightness and palpitations, heart rate irregular', 1),
    ('Kidney stone pain radiating to groin, severity 7 out of 10', 1),
    ('Elevated blood pressure 180/110, mild headache, no stroke signs', 1),
    ('Cellulitis on leg with redness spreading, mild fever', 1),

    # MILD (label 2) — 20 samples
    ('Mild cold and cough for 2 days, no fever', 2),
    ('Sore throat and slight body ache, able to eat normally', 2),
    ('Minor cut on finger, bleeding stopped with pressure', 2),
    ('Skin rash on arm, no other symptoms, itching only', 2),
    ('Mild headache for 1 day, relieved by paracetamol', 2),
    ('Slight nausea after eating, no vomiting or fever', 2),
    ('Common cold with runny nose, mild fatigue, sneezing', 2),
    ('Insect bite with small swelling, no allergic reaction signs', 2),
    ('Back pain from sitting too long, no nerve symptoms', 2),
    ('Mild stomach ache after eating spicy food, passing gas', 2),
    ('Low grade fever 99F, feeling tired, no other symptoms', 2),
    ('Constipation for 2 days, mild discomfort, no bleeding', 2),
    ('Sprained ankle from walking, mild swelling, can bear weight', 2),
    ('Eye redness and mild irritation, no vision change', 2),
    ('Mild ear pain, no discharge, hearing normal', 2),
    ('Anxiety and mild chest tightness, no cardiac history', 2),
    ('Heartburn after meal, relieved partially by antacid', 2),
    ('Bruise on arm after minor bump, no fracture suspected', 2),
    ('Dry skin and mild itching all over body, no rash', 2),
    ('Toothache, mild to moderate, no facial swelling', 2),
]

LABEL_MAP = {0: 'CRITICAL', 1: 'MODERATE', 2: 'MILD'}
texts  = [c[0] for c in complaints]
labels = [c[1] for c in complaints]
print(f'Total samples: {len(complaints)} | Per class: {len(complaints)//3}')

Total samples: 60 | Per class: 20


In [ ]:
# ── Cell 2: Tokenise with DistilBERT tokenizer ──────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODEL_ID = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(texts):
 return tokenizer(texts, padding=True, truncation=True,
                  max_length=128, return_tensors='pt')
tr_txt, va_txt, tr_lbl, va_lbl = train_test_split(texts, labels, test_size=0.2,
random_state=42)

class UrgencyDataset(torch.utils.data.Dataset):
  def __init__(self, texts, labels):
    self.enc = tokenizer(texts, padding=True, truncation=True,
                        max_length=128)
    self.labels = labels
  def __len__(self): return len(self.labels)
  def __getitem__(self, i):
    item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
    item['labels'] = torch.tensor(self.labels[i])
    return item


train_ds = UrgencyDataset(tr_txt, tr_lbl)
val_ds = UrgencyDataset(va_txt, va_lbl)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# ── Cell 3: Fine-tune DistilBERT on GPU ─────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=3)
args = TrainingArguments(
    output_dir='./nlp_model',
    num_train_epochs=8,
    per_device_train_batch_size=8,
    eval_strategy='epoch',   # ← must be here
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=5
)

def compute_metrics(p):
  preds = np.argmax(p.predictions, axis=1)
  return {'accuracy': accuracy_score(p.label_ids, preds)}

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.082601,1.034008,0.416667
2,0.997885,0.935684,0.750000
3,0.844154,0.758119,0.916667
4,0.730132,0.726557,0.750000
5,0.409280,0.527976,0.916667
6,0.317432,0.475150,0.833333
7,0.252182,0.458846,0.916667
8,0.226789,0.457863,0.833333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=48, training_loss=0.5759288196762403, metrics={'train_runtime': 100.7087, 'train_samples_per_second': 3.813, 'train_steps_per_second': 0.477, 'total_flos': 1589637132288.0, 'train_loss': 0.5759288196762403, 'epoch': 8.0})

In [ ]:
# ── Cell 4: Save to Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model('/content/drive/MyDrive/fdp/nlp_model')
tokenizer.save_pretrained('/content/drive/MyDrive/fdp/nlp_model')
print('✅ Model saved to Drive')

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Drive


In [ ]:
# ── Cell 5: spaCy symptom extraction ────────────────────────────
import spacy
nlp_spacy = spacy.load('en_core_web_sm')

SYMPTOM_WORDS = {
    'chest pain', 'breathlessness', 'fever', 'vomiting', 'headache',
    'dizziness', 'nausea', 'weakness', 'cough', 'rash', 'swelling',
    'paralysis', 'seizure', 'unconscious', 'sweating', 'bleeding',
    'burning', 'fatigue', 'palpitations', 'wheezing', 'confusion'
}

def extract_symptoms(text):
    found = [w for w in SYMPTOM_WORDS if w in text.lower()]
    return list(set(found))

In [ ]:
# ── Cell 6: analyze_complaint() — fixed ──────────────────────────
LABEL_MAP = {0: 'CRITICAL', 1: 'MODERATE', 2: 'MILD'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)  # ensure model is on correct device

def analyze_complaint(text):
    enc = tokenizer(
        [text], padding=True, truncation=True,
        max_length=128, return_tensors='pt'
    )
    # ── move all input tensors to same device as model ──
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
    pred_id  = int(torch.argmax(logits))
    urgency  = LABEL_MAP[pred_id]
    symptoms = extract_symptoms(text)
    return {'symptoms': symptoms, 'urgency': urgency}

# ── Test ─────────────────────────────────────────────────────────
print(analyze_complaint('Severe chest pain and breathlessness since 2 hours'))
print(analyze_complaint('Mild cold and cough, no fever'))
print(analyze_complaint('High fever with vomiting and dehydration'))

{'symptoms': ['chest pain', 'breathlessness'], 'urgency': 'MODERATE'}
{'symptoms': ['cough', 'fever'], 'urgency': 'MILD'}
{'symptoms': ['fever', 'vomiting'], 'urgency': 'MODERATE'}


In [ ]:
# ── Cell 7: Save nlp_module.py for Day 5 ─────────────────────────
nlp_module_code = '''
# Day 3 — NLP Module
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = '/content/drive/MyDrive/fdp/nlp_model'
LABEL_MAP  = {0: 'CRITICAL', 1: 'MODERATE', 2: 'MILD'}
SYMPTOM_WORDS = {
    'chest pain', 'breathlessness', 'fever', 'vomiting', 'headache',
    'dizziness', 'nausea', 'weakness', 'cough', 'rash', 'swelling',
    'paralysis', 'seizure', 'unconscious', 'sweating', 'bleeding',
    'burning', 'fatigue', 'palpitations', 'wheezing', 'confusion'
}

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

nlp_spacy = spacy.load('en_core_web_sm')

def extract_symptoms(text):
    return list({w for w in SYMPTOM_WORDS if w in text.lower()})

def analyze_complaint(text):
    enc = tokenizer([text], padding=True, truncation=True,
                    max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred_id  = int(torch.argmax(logits))
    return {
        'symptoms': extract_symptoms(text),
        'urgency' : LABEL_MAP[pred_id]
    }
'''

with open('/content/drive/MyDrive/fdp/nlp_module.py', 'w') as f:
    f.write(nlp_module_code)

print('✅ nlp_module.py saved to Drive')

✅ nlp_module.py saved to Drive
